# Naive RAG Walkthrough
투자 유튜브 -> pgvector -> gpt-4o-mini.

## 0. 원본 데이터 수집 파이프라인 (참고)

**파이프라인**: YouTube URL → `yt-dlp`로 m4a 오디오 다운로드 → (선택) `ffmpeg`로 2배속 → OpenAI Whisper API로 한국어 transcribe → PostgreSQL `contents.transcript`에 저장.

In [ ]:
# 참고용 — 실제 실행 안 함 (DB에 이미 적재됨)
# 실행하려면: `uv pip install yt-dlp openai` + ffmpeg 설치 후 RUN_INGEST = True

RUN_INGEST = False

from pathlib import Path
import subprocess
from datetime import datetime


def download_audio(url: str, data_dir: Path) -> tuple[dict, Path]:
    """yt-dlp로 비디오에서 m4a 오디오만 추출."""
    import yt_dlp

    data_dir.mkdir(parents=True, exist_ok=True)
    opts = {
        "format": "bestaudio/best",
        "outtmpl": str(data_dir / "%(id)s.%(ext)s"),
        "postprocessors": [
            {"key": "FFmpegExtractAudio", "preferredcodec": "m4a"},
        ],
        "quiet": True,
        "no_warnings": True,
    }
    with yt_dlp.YoutubeDL(opts) as ydl:
        info = ydl.extract_info(url, download=True)
    audio_path = next(data_dir.glob(f"{info['id']}.*"))
    return info, audio_path


def speed_up_audio(source: Path, factor: float = 2.0) -> Path:
    """ffmpeg atempo로 N배속 가속 (Whisper 비용 절감)."""
    if factor == 1.0:
        return source
    output = source.with_name(f"{source.stem}.x{factor}{source.suffix}")
    subprocess.run(
        ["ffmpeg", "-y", "-loglevel", "error", "-i", str(source),
         "-filter:a", f"atempo={factor}", "-vn", str(output)],
        check=True,
    )
    return output


def transcribe(audio_path: Path, api_key: str) -> str:
    """OpenAI Whisper API로 한국어 transcribe."""
    from openai import OpenAI

    client = OpenAI(api_key=api_key)
    with audio_path.open("rb") as f:
        text = client.audio.transcriptions.create(
            model="whisper-1",
            response_format="text",
            file=f,
            language="ko",
        )
    return text.strip() if isinstance(text, str) else str(text).strip()


def save_to_postgres(
    database_url: str,
    *,
    persona_id: str,
    video_id: str,
    url: str,
    title: str,
    published_at: datetime,
    transcript_text: str,
) -> None:
    """contents 테이블에 INSERT. 메인 프로젝트 schema 준수."""
    from sqlalchemy import create_engine, text

    engine = create_engine(database_url)
    with engine.begin() as conn:
        conn.execute(
            text("""
                INSERT INTO contents
                  (id, persona_id, source_type, source_url, title,
                   published_at, transcript, ingested_at)
                VALUES
                  (gen_random_uuid(), :persona_id, 'youtube', :url, :title,
                   :published_at, :transcript, NOW())
                ON CONFLICT (source_url) DO NOTHING
            """),
            {
                "persona_id": persona_id,
                "url": url,
                "title": title,
                "published_at": published_at,
                "transcript": transcript_text,
            },
        )


if RUN_INGEST:
    from naive_rag.config import Settings

    settings = Settings()
    url = "https://www.youtube.com/watch?v=YOUR_VIDEO_ID"
    data_dir = Path("data/audio")

    info, audio = download_audio(url, data_dir)
    audio_fast = speed_up_audio(audio, factor=2.0)  # 비용/시간 반으로
    transcript_text = transcribe(audio_fast, settings.openai_api_key)
    save_to_postgres(
        settings.main_database_url,
        persona_id="sosumonkey",  # personas 테이블의 ID
        video_id=info["id"],
        url=info["webpage_url"],
        title=info["title"],
        published_at=datetime.strptime(info["upload_date"], "%Y%m%d"),
        transcript_text=transcript_text,
    )
    print(f"saved: {info['title']} ({len(transcript_text)} chars)")

In [1]:
from naive_rag.loader import load_transcripts
docs = load_transcripts(limit=3)
print(f"Loaded {len(docs)} documents")
docs[0].metadata, docs[0].page_content[:300]

Loaded 3 documents


({'content_id': '57e3e4ed-ae08-4eae-9f13-daecf96554d1',
  'title': '[오늘의 미국주식뉴스] 새롭게 편입된 미주은 탑픽은? / 포트폴리오 비중 조절의 기준 / 빅테크 4기업 어닝 실적 이것만 보면 됩니다 / 낸시 탱글러 선정 최고의 AI 수혜주',
  'channel': '미국주식으로 은퇴하기',
  'published_at': '2026-04-30 00:00:00'},
 '안녕하세요. 미국 주식론 대학의 미준입니다. 4월 30일 오늘의 미국 주식론 뉴스를 시작합니다. 오늘의 변함없이 미국 주식 투자의 정석을 지키고 있는 미준의 뉴스를 환영합니다. 오늘은 목요일입니다. 일주일에 7일이 있는데요. 각 요일마다 가지고 있는 이미지, 느낌이 조금씩 다르잖아요. 여러분들께 목요일은 어떤 느낌으로 다가오는지 궁금합니다. 저 같은 경우는 금요일이 되면 왠지 좀 나사가 풀리고요. 조금은 여유를 가져도 되겠다 이런 생각이 들거든요. 그러다 보니까 목요일이 되면요. 왠지 좀 긴장이 됩니다. 하루를 굉장히 열심히 살아야')

## Split

In [2]:
from naive_rag.splitter import split_documents
chunks = split_documents(docs)
print(f"{len(docs)} docs -> {len(chunks)} chunks")
chunks[0].page_content[:200], chunks[0].metadata

3 docs -> 84 chunks


('안녕하세요. 미국 주식론 대학의 미준입니다. 4월 30일 오늘의 미국 주식론 뉴스를 시작합니다. 오늘의 변함없이 미국 주식 투자의 정석을 지키고 있는 미준의 뉴스를 환영합니다. 오늘은 목요일입니다. 일주일에 7일이 있는데요. 각 요일마다 가지고 있는 이미지, 느낌이 조금씩 다르잖아요. 여러분들께 목요일은 어떤 느낌으로 다가오는지 궁금합니다. 저 같은 경우는',
 {'content_id': '57e3e4ed-ae08-4eae-9f13-daecf96554d1',
  'title': '[오늘의 미국주식뉴스] 새롭게 편입된 미주은 탑픽은? / 포트폴리오 비중 조절의 기준 / 빅테크 4기업 어닝 실적 이것만 보면 됩니다 / 낸시 탱글러 선정 최고의 AI 수혜주',
  'channel': '미국주식으로 은퇴하기',
  'published_at': '2026-04-30 00:00:00'})

## Embed (single query for inspection)

In [3]:
from naive_rag.embeddings import build_embeddings
emb = build_embeddings()
vec = emb.embed_query("매크로 지표")
print(f"dim={len(vec)}, first 5={vec[:5]}")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

dim=1024, first 5=[-0.02018229104578495, -0.01778406836092472, -0.05371119827032089, -0.02455992065370083, 0.0004122735117562115]


## Retrieve

In [6]:
from naive_rag.retriever import build_retriever
retriever = build_retriever()
hits = retriever.invoke("ai 수혜주는?")
for i, h in enumerate(hits):
    print(f"--- hit {i} (channel={h.metadata.get('channel')}) ---")
    print(h.page_content[:200])

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

--- hit 0 (channel=소수몽키) ---
. 그 외에도 제가 말씀드렸던 것처럼, 아마존의 자체 반도체, 구글의 자체 반도체, 퀄컴 등의 유명한 그런 테크 기업의 반도체, 대부분 ARM 설계한 ARM 기반의 반도체를 쓰기 때문에, 뒤에서 오는 또 하나의 수혜주로 볼 수 있겠죠? 그래서 저 같은 경우는 이 삼총사가 CPU 수혜주로 좀 보여질 것 같습니다. 인텔, AMD, ARM 이렇게 보시면 될 것 
--- hit 1 (channel=소수몽키) ---
. 물론, 그 당시 엔비디아는 독주였고, 지금은 뭐, 인텔, AMD, 아마존 등등, 독주의 개념은 아니기 때문에, 제가 봤을 땐 그 정도까지는 아니겠지만, 그래도, 기대감이 여기로 또 옮겨 붙고 있다, 좀 생각해보시면 좋을 것 같습니다. 참고로, 아마존이나, 구글이나, 이런 빅테크들이 만드는 자체 반도체 CPU는 다, 반도체 ARM이라는 회사의 기반으로 만
--- hit 2 (channel=소수몽키) ---
해명했으나, 앞으로 이쪽으로 갈 거라는 거에는 이견이 없을 것 같습니다 사실상 이제 비싼 요금제를 써야지만 AI 에이전트를 쓸 수 있는데 그것도 지금 사람들이 수요가 폭증하고 있다 이렇게 보시면 될 것 같고 마이크로소프트도 본인들 모델, 마이크로소프트 제품 구독할 때 가장 비싼 요금제 쓰는 구독자만 엔트로픽 AI 에이전트 기능 쓸 수 있다 이런 식으로 고객
--- hit 3 (channel=소수몽키) ---
. 자, 그래서 어쨌든 제가 봤을 땐 엔트로픽의 미래를 엄청 밝게 보고 지금 빅테크들이 막 압도적으로 큰 돈을 쏟아붓고 있다 이렇게 보셔야 될 것 같은데 실제로 저도 지금 냉정하게 말하면 엔트로픽에다가 구동료를 몇백 불 이상 쓰고 있기 때문에 지금 이렇게 직원들 거까지 다 쓰고 있기 때문에 예를 들어서 제가 지금 채치피티랑 재미나이는 기본요금을 쓰고 있는데


## Generate

In [7]:
from naive_rag.chain import build_chain
chain = build_chain()
answer = chain.invoke("ai 수혜주는?")
print(answer)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

AI 수혜주로는 인텔, AMD, ARM이 언급되었습니다. 이들은 CPU 반도체 분야에서 수혜를 받을 것으로 보입니다. 또한, 아마존과 구글의 자체 반도체도 ARM 기반으로 만들어지기 때문에 이들 역시 수혜주로 고려될 수 있습니다.


## Run all 10 eval questions

In [8]:
questions = [
    "최근 AI 투자 전쟁에서 주목한 수혜주는 무엇인가?",
    "AI 에이전트가 반도체 주도주 판도를 어떻게 바꾸고 있나?",
    "트럼프 순방과 젠슨황의 깐부회동이 증시에 미친 영향은?",
    "2026 버블 시나리오에서 본격 버블은 시작됐다고 보나?",
    "최근 폭락한 미국 초우량주 중 매수 후보로 본 종목은?",
    "헬스케어 섹터(유나이티드헬스, 노보노 등)에 대한 시각은?",
    "인텔, AMD, ARM 등 반도체 종목에 대한 평가는?",
    "GTC 워싱턴 행사에서 엔비디아가 발표한 신규 사업 영역은?",
    "2030년까지 컴퓨팅 용량 확장과 AI 인프라 전망은?",
    "빅테크(아마존, 구글, 엔비디아) 중 AI 투자에 가장 적극적인 곳은?",
]
for i, q in enumerate(questions, 1):
    print(f"\n=== Q{i}. {q}")
    print(chain.invoke(q))


=== Q1. 최근 AI 투자 전쟁에서 주목한 수혜주는 무엇인가?
최근 AI 투자 전쟁에서 주목한 수혜주는 아마존과 구글입니다. 이 두 기업은 엔트로픽 AI에이전트 붐과 관련하여 계속해서 주목받고 있으며, 실적 발표에서 미끄러질 경우 추가 매수 대응을 계획하고 있다고 언급되었습니다. 또한, 엔비디아도 AI 인프라 붐의 열풍 속에서 기대감을 받고 있습니다.

=== Q2. AI 에이전트가 반도체 주도주 판도를 어떻게 바꾸고 있나?
AI 에이전트가 반도체 주도주 판도를 바꾸고 있는 이유는 AI 붐이 CPU로 확대되고 있기 때문입니다. 기존에는 엔비디아의 GPU가 주도했으나, AI 에이전트 시대에 접어들면서 CPU와 메모리의 중요성이 증가하고 있습니다. 특히, 빅테크 기업들이 자체 반도체를 개발하고 CPU를 대량 구매하는 등의 움직임이 나타나면서 CPU 주식들이 상승세를 보이고 있습니다. 이러한 변화는 컴퓨팅 병목 현상이 GPU에서 CPU로 이동하고 있음을 나타내며, 인텔, AMD, ARM과 같은 CPU 반도체 기업들이 수혜를 받을 것으로 예상됩니다.

=== Q3. 트럼프 순방과 젠슨황의 깐부회동이 증시에 미친 영향은?
트럼프의 아시아 APEC 순방과 젠슨황의 깐부회동은 증시에 긍정적인 영향을 미쳤습니다. 트럼프는 순방 후 수조 달러의 투자 유치를 강조하며 미국 주식 시장에 대한 자신감을 표명했습니다. 또한, 젠슨황은 한국을 방문하여 기업 총수들과 회동하고 엔비디아의 반도체 공급 계획을 발표하는 등, 이러한 활동들이 증시를 끌어올리는 데 기여했습니다. 전반적으로 증시는 상승세를 보였고, 이러한 이슈들이 투자자들에게 긍정적인 신호로 작용한 것으로 보입니다.

=== Q4. 2026 버블 시나리오에서 본격 버블은 시작됐다고 보나?
주어진 자료로는 알 수 없습니다.

=== Q5. 최근 폭락한 미국 초우량주 중 매수 후보로 본 종목은?
주어진 자료로는 알 수 없습니다.

=== Q6. 헬스케어 섹터(유나이티드헬스, 노보노 등)에 대한 시각은?
헬스케어 섹터, 특히 유

## Ragas Evaluation

4 metrics: **Faithfulness**, **AnswerRelevancy**, **LLMContextPrecisionWithoutReference**, **ContextEntityRecall**.

ContextEntityRecall은 reference(정답)에서 entity를 추출해 contexts에 얼마나 들어있는지 본다. 아래 `eval_data` 리스트에서 각 질문의 `reference` 값을 직접 입력. Reference는 핵심 entity(종목명, 인물, 지표명 등)가 들어있는 1-3문장이 좋다.

Judge LLM은 gpt-4o-mini, judge embeddings는 bge-m3.

In [9]:
from datasets import Dataset
from langchain_openai import ChatOpenAI
from ragas import evaluate
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import (
    AnswerRelevancy,
    ContextEntityRecall,
    Faithfulness,
    LLMContextPrecisionWithoutReference,
)

from naive_rag.chain import build_chain
from naive_rag.config import Settings
from naive_rag.embeddings import build_embeddings
from naive_rag.retriever import build_retriever

# 5개 소수몽키 transcript 내용 기반으로 작성된 reference 정답.
# 직접 수정해서 본인 시각으로 재작성해도 됨.
eval_data = [
    {
        "question": "최근 AI 투자 전쟁에서 주목한 수혜주는 무엇인가?",
        "reference": "구글과 아마존이 엔트로픽에 묻고 더블로 베팅한 빅테크, 자체 반도체(구글 TPU 8세대, 아마존 그래비톤), 그리고 데이터센터 인프라주(GE Vernova 가스터빈, 유나이티드 렌탈·캐터필러 중장비, 이튼·컨스털레이션 전력, 버티브 냉각, 컴포트 시스템 시공)가 수혜주로 언급되었다.",
    },
    {
        "question": "AI 에이전트가 반도체 주도주 판도를 어떻게 바꾸고 있나?",
        "reference": "AI 에이전트 시대에는 다양한 업무를 처리해야 하므로 CPU(인텔, AMD, ARM)의 중요도가 GPU 못지않게 커진다는 평가가 모건스탠리 보고서로 확산되었다. 메타가 아마존의 자체 CPU 그래비톤을 수천만 개 구매한 사건이 이를 뒷받침한다.",
    },
    {
        "question": "트럼프 순방과 젠슨황의 깐부회동이 증시에 미친 영향은?",
        "reference": "트럼프 아시아 순방으로 한미·미일·미중 협력 패키지가 발표되며 미국 주식회사 전체가 투자 유치 수혜를 받았다. 젠슨황의 깐부회동에서 엔비디아는 한국 정부·삼성·현대차·SK·네이버에 반도체 26만 장(약 10-15조원)을 판매했고, GTC 워싱턴 개최로 엔비디아가 국가급 AI 프로젝트의 핵심으로 부상했다.",
    },
    {
        "question": "2026 버블 시나리오에서 본격 버블은 시작됐다고 보나?",
        "reference": "신한투자증권 김성환 연구원의 리포트에 따르면 본격 버블은 2026-2027년에 도래하며 현재는 아직 버블이 아니다. 1995년 다펌 장, 2016년 클라우드 사이클처럼 기술 혁신 주도 사이클은 약 5년 지속되며, PER만으로는 판단이 빡빡해지는 단계라 기술적 분석 병행이 필요하다.",
    },
    {
        "question": "최근 폭락한 미국 초우량주 중 매수 후보로 본 종목은?",
        "reference": "노보노디스크(비만약 위고비, 70% 가까이 폭락)와 유나이티드 헬스케어(미국 최대 민간의료보험, 사상 최악의 폭락)가 한국인 순매수 상위에 오르며 줍줍 대상이 되었다. 다만 보잉처럼 5년 박스권에 갇히는 리스크도 함께 거론되었다.",
    },
    {
        "question": "헬스케어 섹터(유나이티드헬스, 노보노 등)에 대한 시각은?",
        "reference": "트럼프 행정부의 최대 피해주는 제약사와 의료보험사(PBM 중간 유통 마진 타겟). 노보노디스크는 비만약 경쟁 심화로 가이던스 빅배스 의심, 유나이티드 헬스케어는 의료비 폭증으로 실적 쇼크. 다만 1기 때도 약값 누르기 시도가 결국 실패했다는 전설이 있어 회복 사이클 기대도 함께 존재한다.",
    },
    {
        "question": "인텔, AMD, ARM 등 반도체 종목에 대한 평가는?",
        "reference": "인텔은 트럼프 백악관 회동 이후 20불에서 80불로 4배 상승했고 2000년 닷컴버블 고점을 처음 돌파했다. CEO가 'AI 에이전트 시대 CPU 재평가' 구조적 변화를 직접 언급했다. AMD는 CPU 가격 인상과 공급 부족, ARM은 박스권 돌파와 자체 CPU 판매 선언으로 삼총사 모두 신고가를 기록했다.",
    },
    {
        "question": "GTC 워싱턴 행사에서 엔비디아가 발표한 신규 사업 영역은?",
        "reference": "엔비디아는 GTC 워싱턴에서 AI를 미국의 아폴로 프로젝트급으로 선언하고, 양자컴퓨터·슈퍼컴퓨터, 노키아 지분 인수를 통한 6G 통신, 사이버보안, 팔란티어와의 데이터 분석 협업, 자율주행, 일라이 릴리와의 신약 개발, 휴머노이드 로봇 등 전방위 사업 확장을 발표했다.",
    },
    {
        "question": "2030년까지 컴퓨팅 용량 확장과 AI 인프라 전망은?",
        "reference": "OpenAI는 2025년 10GW 목표 중 8GW 확보 완료, 2030년까지 30GW를 새 목표로 발표했다. 반도체·전력주·메모리·광통신이 모두 엮여 있어 AI 인프라 붐은 갈 길이 더 많다(남은 22GW). 현재까지 한 게 8GW이므로 앞으로 할 게 더 많다는 의미.",
    },
    {
        "question": "빅테크(아마존, 구글, 엔비디아) 중 AI 투자에 가장 적극적인 곳은?",
        "reference": "구글이 엔트로픽에 기존 33억불 대비 10배 이상인 400억불(즉시 100억+성과 300억)을 추가 베팅하며 가장 적극적이었고, 아마존도 기존 80억불의 3배 이상인 250억불(즉시 50억+성과 200억)을 추가 투자했다. 엔비디아는 오픈 AI 연합으로 두 진영(엔트로픽 vs 오픈 AI) 경쟁이 핵심 구도가 되었다.",
    },
]

# Reference 미입력 항목 체크
missing = [d["question"] for d in eval_data if not d["reference"].strip()]
if missing:
    raise ValueError(
        f"{len(missing)}개 질문의 reference가 비어 있다. eval_data를 채운 뒤 다시 실행:\n"
        + "\n".join(f"  - {q}" for q in missing)
    )

retriever = build_retriever()
chain = build_chain()
rows = [
    {
        "user_input": d["question"],
        "reference": d["reference"],
        "response": chain.invoke(d["question"]),
        "retrieved_contexts": [c.page_content for c in retriever.invoke(d["question"])],
    }
    for d in eval_data
]

settings = Settings()
result = evaluate(
    Dataset.from_list(rows),
    metrics=[
        Faithfulness(),
        AnswerRelevancy(),
        LLMContextPrecisionWithoutReference(),
        ContextEntityRecall(),
    ],
    llm=LangchainLLMWrapper(
        ChatOpenAI(
            model=settings.llm_model,
            api_key=settings.openai_api_key,
            temperature=0,
        )
    ),
    embeddings=LangchainEmbeddingsWrapper(build_embeddings()),
)
result.to_pandas()

/var/folders/_3/4lsqsqp92l9_2rwbg43vfbpr0000gn/T/ipykernel_44686/3897902304.py:6: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import (
/var/folders/_3/4lsqsqp92l9_2rwbg43vfbpr0000gn/T/ipykernel_44686/3897902304.py:6: DeprecationWarning: Importing ContextEntityRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextEntityRecall
  from ragas.metrics import (
/var/folders/_3/4lsqsqp92l9_2rwbg43vfbpr0000gn/T/ipykernel_44686/3897902304.py:6: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

/var/folders/_3/4lsqsqp92l9_2rwbg43vfbpr0000gn/T/ipykernel_44686/3897902304.py:92: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  llm=LangchainLLMWrapper(


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

/var/folders/_3/4lsqsqp92l9_2rwbg43vfbpr0000gn/T/ipykernel_44686/3897902304.py:99: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  embeddings=LangchainEmbeddingsWrapper(build_embeddings()),


Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,llm_context_precision_without_reference,context_entity_recall
0,최근 AI 투자 전쟁에서 주목한 수혜주는 무엇인가?,"[자, 이번 주 가장 중요한 이슈 있죠? 바로 슈퍼위크입니다. 빅테크 5개가 실적 ...",최근 AI 투자 전쟁에서 주목한 수혜주는 아마존과 구글입니다. 이 두 기업은 엔트로...,"구글과 아마존이 엔트로픽에 묻고 더블로 베팅한 빅테크, 자체 반도체(구글 TPU 8...",1.000000,0.901662,1.0,0.250000
1,AI 에이전트가 반도체 주도주 판도를 어떻게 바꾸고 있나?,"[. 그래서, 이번에도 모건센니가 보고서를 내면서, AI 붐이 드디어 CPU로 넘어...",AI 에이전트가 반도체 주도주 판도를 바꾸고 있는 이유는 AI 붐이 CPU로 확대되...,"AI 에이전트 시대에는 다양한 업무를 처리해야 하므로 CPU(인텔, AMD, ARM...",1.000000,0.794173,1.0,0.800000
2,트럼프 순방과 젠슨황의 깐부회동이 증시에 미친 영향은?,"[아주 정리해보도록 하겠습니다. 자, 일단은 여기 그림 보시는 것처럼요. 오른쪽 그...",트럼프의 아시아 APEC 순방과 젠슨황의 깐부회동은 증시에 긍정적인 영향을 미쳤습니...,트럼프 아시아 순방으로 한미·미일·미중 협력 패키지가 발표되며 미국 주식회사 전체가...,1.000000,0.664252,1.0,0.187500
3,2026 버블 시나리오에서 본격 버블은 시작됐다고 보나?,[. 알겠습니다. 제가 들었다가 말씀하신 거에 대해서 궁금한 게 우리가 사실 제대로...,주어진 자료로는 알 수 없습니다.,신한투자증권 김성환 연구원의 리포트에 따르면 본격 버블은 2026-2027년에 도래...,0.000000,0.000000,1.0,0.000000
4,최근 폭락한 미국 초우량주 중 매수 후보로 본 종목은?,[. 이해 되죠? 무슨 말인지. 그러니까 싸고 좋고 많이 빠진 거 아는데 이게 다시...,주어진 자료로는 알 수 없습니다.,"노보노디스크(비만약 위고비, 70% 가까이 폭락)와 유나이티드 헬스케어(미국 최대 ...",0.000000,0.000000,1.0,0.500000
5,"헬스케어 섹터(유나이티드헬스, 노보노 등)에 대한 시각은?",[. 미국의 최대 민간의료보험회사 유나이티드 헬스케어. 줄여서 유나헤리라고 많이 표...,"헬스케어 섹터, 특히 유나이티드 헬스케어와 노보노디스크에 대한 시각은 현재 부정적입...",트럼프 행정부의 최대 피해주는 제약사와 의료보험사(PBM 중간 유통 마진 타겟). ...,1.000000,0.000000,1.0,0.250000
6,"인텔, AMD, ARM 등 반도체 종목에 대한 평가는?","[. 자, 그래서 오늘은 그 와중에 반도체 얘기를 안 하셨습니다. 요즘 너무 핫하죠...","인텔, AMD, ARM은 최근 CPU의 부활 재평가와 함께 주목받고 있으며, 이들 ...",인텔은 트럼프 백악관 회동 이후 20불에서 80불로 4배 상승했고 2000년 닷컴버...,0.833333,0.679115,1.0,0.307692
7,GTC 워싱턴 행사에서 엔비디아가 발표한 신규 사업 영역은?,[. 약간 가려졌는데 정말 곡괭이 파는 나중에 그냥 경제학 교과서 주식시장의 교과서...,주어진 자료로는 알 수 없습니다.,"엔비디아는 GTC 워싱턴에서 AI를 미국의 아폴로 프로젝트급으로 선언하고, 양자컴퓨...",0.000000,0.000000,0.5,0.250000
8,2030년까지 컴퓨팅 용량 확장과 AI 인프라 전망은?,[이번에 보고서를 자체적으로 발표를 하면서 자 우리가 컴퓨팅 용량 2025년에 10...,"2030년까지 컴퓨팅 용량은 30GW로 확장할 계획이며, 현재 8GW를 확보한 상태...","OpenAI는 2025년 10GW 목표 중 8GW 확보 완료, 2030년까지 30G...",1.000000,0.882590,1.0,0.714286
9,"빅테크(아마존, 구글, 엔비디아) 중 AI 투자에 가장 적극적인 곳은?","[자, 이번 주 가장 중요한 이슈 있죠? 바로 슈퍼위크입니다. 빅테크 5개가 실적 ...",주어진 자료로는 알 수 없습니다.,구글이 엔트로픽에 기존 33억불 대비 10배 이상인 400억불(즉시 100억+성과 ...,0.000000,0.000000,1.0,0.571429
